In [ ]:
import pandas as pd
import warnings
import platform
from sqlalchemy import create_engine
import pandas as pd
import numpy as np
import pymysql
import datetime as dt
import os
import json
import platform
from googleapiclient.discovery  import build
from google.oauth2 import service_account
import psycopg2
import pandas as pd
import numpy as np
import datetime 
import pymysql
import yagmail
import time
import platform
from sqlalchemy import create_engine
from googleapiclient.discovery import build
from google.oauth2 import service_account
from datetime import timedelta,datetime
from datetime import datetime
from dateutil.relativedelta import relativedelta
### SQL trying

import requests
import json
import pandas as pd
import time
import platform
import datetime
import psycopg2
import yagmail
from datetime import datetime
from dateutil.relativedelta import relativedelta
from io import StringIO
from sqlalchemy import create_engine

import traceback
from langchain_openai import ChatOpenAI
from sqlalchemy import create_engine,text
from dotenv import load_dotenv
import decimal
import math
import pytz
from datetime import datetime

# import gspread
# from oauth2client.service_account import ServiceAccountCredentials
from google.oauth2 import service_account
from googleapiclient.discovery import build


In [ ]:
# FL = {'location': r"C:\Documents\Python_Script"}

FL = {'location': r"C:\Users\Amit Singh\Documents\Python_Scripts"}

In [ ]:
#For Voylla DB Cred
if platform.system()=='Windows':
    with open(r"%s\Voylla_Cred.txt" % FL['location'],'r') as f:
        lines = f.readlines()
        lines = [item.strip() for item in lines]
        Voylla_config = {
            'user':  lines[1],
            'password': lines[2],
            'host': lines[0],
            'database': lines[3],
            'port': int(lines[4])
        }
    
elif platform.system()=='Linux':
    with open(r'/home/misauto/Python_Scripts/Voylla_Cred.txt') as f:
        lines = f.readlines()
        lines = [item.strip() for item in lines]
        Voylla_config = {
            'user':  lines[1],
            'password': lines[2],
            'host': lines[0],
            'database': lines[3],
            'port': int(lines[4])
        }

# for gpt api key
if platform.system()=='Windows':
    with open(r"%s\Gpt_api_key.txt" % FL['location'],'r') as f:
        lines = f.readlines()
        lines = [item.strip() for item in lines]
        Gpt_api = {
            'model_name':  lines[0],
            'api': lines[1]
        }
    
elif platform.system()=='Linux':
    with open(r'/home/misauto/Python_Scripts/Gpt_api_key.txt') as f:
        lines = f.readlines()
        lines = [item.strip() for item in lines]
        Gpt_api = {
            'model_name':  lines[0],
            'api': lines[1]
        }



if platform.system()=='Windows':
    with open(r"%s\Claude_api_key.txt" % FL['location'],'r') as f:
        lines = f.readlines()
        lines = [item.strip() for item in lines]
        Claude_api = {
            'model_name':  lines[0],
            'api': lines[1]
        }
    
elif platform.system()=='Linux':
    with open(r'/home/misauto/Python_Scripts/Claude_api_key.txt') as f:
        lines = f.readlines()
        lines = [item.strip() for item in lines]
        Claude_api = {
            'model_name':  lines[0],
            'api': lines[1]
        }

In [ ]:
# LLM_TEMPERATURE = 0

# # MODEL_NAME = "gpt-4.1-mini"

# # Extract values

# db_host = Voylla_config["host"]
# db_port = Voylla_config["port"]
# db_name = Voylla_config["database"]
# db_user = Voylla_config["user"]
# db_password = Voylla_config["password"]

# api_key=Gpt_api["api"]
# MODEL_NAME= Gpt_api["model_name"]


# def get_llm():
#     return ChatOpenAI(model=MODEL_NAME, temperature=LLM_TEMPERATURE, request_timeout=120, max_retries=3,api_key=api_key)

# llm = get_llm()


# # DB connection
# def get_engine_and_schema():
    
#     try:
#         engine = create_engine(
#             f"postgresql+psycopg2://{db_user}:{db_password}@{db_host}:{db_port}/{db_name}",
#             pool_pre_ping=True,
#             pool_recycle=3600,
#             pool_size=5,
#             max_overflow=10
#         )

#         # ⭐ test connection immediately
#         with engine.connect() as conn:
#             print("✅ Database connection successful")

#         return engine


#     except Exception as e:
#         print(f"❌ Unexpected error while creating engine: {e}")
#         return None



# engine = get_engine_and_schema()
# if engine is None:
#     print("Stopping script due to DB failure")
#     exit()


In [ ]:
import httpx
import anthropic

LLM_TEMPERATURE = 0

# Extract values
db_host = Voylla_config["host"]
db_port = Voylla_config["port"]
db_name = Voylla_config["database"]
db_user = Voylla_config["user"]
db_password = Voylla_config["password"]

api_key = Claude_api["api"]
MODEL_NAME = "claude-haiku-4-5-20251001"  # Cost optimized: ~8x cheaper than Sonnet

def get_llm():
    return anthropic.Anthropic(
        api_key=api_key,
        http_client=httpx.Client(timeout=httpx.Timeout(180.0))
    )

llm = get_llm()

# DB connection stays exactly the same
def get_engine_and_schema():
    try:
        engine = create_engine(
            f"postgresql+psycopg2://{db_user}:{db_password}@{db_host}:{db_port}/{db_name}",
            pool_pre_ping=True,
            pool_recycle=3600,
            pool_size=5,
            max_overflow=10
        )
        with engine.connect() as conn:
            print("✅ Database connection successful")
        return engine
    except Exception as e:
        print(f"❌ Unexpected error while creating engine: {e}")
        return None

engine = get_engine_and_schema()
if engine is None:
    print("Stopping script due to DB failure")
    exit()


In [ ]:
query = """
SELECT DISTINCT "Brand"
FROM voylla."Blinkit_Ads_Report"
"""

df = pd.read_sql(query, engine)

Brands = df["Brand"].dropna().unique().tolist()

print(Brands)
# Brands=['Voylla']

In [ ]:
import re
import json
def extract_json(response: str) -> list:
    """
    Extracts JSON array from LLM response.
    Handles scratchpad lines, markdown fences, and partial responses.
    """
    response = re.sub(r'```(?:json)?', '', response).strip()
    
    json_start = response.find('[')
    if json_start < 0:
        raise ValueError("No JSON array found in response")
    response = response[json_start:]
    
    json_end = response.rfind(']') + 1
    if json_end == 0:
        raise ValueError("No closing bracket found in response")
    response = response[:json_end]
    
    try:
        return json.loads(response)
    except json.JSONDecodeError as e:
        response_fixed = re.sub(r',\s*([}\]])', r'\1', response)
        try:
            return json.loads(response_fixed)
        except json.JSONDecodeError:
            raise ValueError(f"Could not parse JSON from response: {e}")

In [ ]:
# import json
# import re

# def extract_json(text):

#     # remove markdown
#     text = text.replace("```json", "").replace("```", "")

#     # find first JSON array
#     match = re.search(r"\[.*\]", text, re.S)

#     if not match:
#         raise ValueError("No JSON found in LLM response")

#     cleaned = match.group(0)

#     # remove trailing commas
#     cleaned = re.sub(r",\s*}", "}", cleaned)
#     cleaned = re.sub(r",\s*]", "]", cleaned)

#     return json.loads(cleaned)

In [ ]:

from sqlalchemy import text
from datetime import datetime


In [ ]:
def aggregate_window(df, days):
    """
    Aggregate performance for given window
    """

    agg = df.groupby(['Campaign ID', 'Targeting Value']).agg({
        'spend': 'sum',
        'total_sales': 'sum',
        'impressions': 'sum',
        'total_atc': 'sum',
        'total_units': 'sum',
        'most_viewed_position': 'median',
        'Campaign Name': 'last'
    }).reset_index()

    agg[f'roas_{days}d'] = (agg['total_sales'] / agg['spend']).replace(
        [float('inf'), -float('inf')], 0
    ).fillna(0).round(2)

    agg[f'ctr_{days}d'] = (agg['total_atc'] / agg['impressions']).fillna(0).round(4)

    agg = agg.rename(columns={
        'Campaign ID': 'campaign_id',
        'Targeting Value': 'targeting',
        'most_viewed_position': 'position',
        'spend': f'spend_{days}d',
        'total_sales': f'sales_{days}d'
    })

    return agg


In [ ]:
import pandas as pd

def resolve_impl(user_impl):
    if user_impl is None:
        return "Unknown"
    if isinstance(user_impl, str):
        val = user_impl.strip().lower()
        if val == 'true':
            return "Implemented"
        elif val == 'false':
            return "Not implemented"
        else:
            return "Unknown"
    try:
        if pd.isna(user_impl):
            return "Unknown"
    except (TypeError, ValueError):
        pass
    return "Implemented" if bool(user_impl) else "Not implemented"


def detect_oscillation(history_list):
    """
    Detect CPM ping-pong: alternating INCREASE_CPM <-> DECREASE_CPM.
    Requires at least 3 alternating direction-changing actions.
    Returns (is_loop: bool, description: str).
    """
    if len(history_list) < 3:
        return False, ""
    change_actions = {"INCREASE_CPM", "DECREASE_CPM"}
    change_seq = [h.get("action", "") for h in history_list if h.get("action", "") in change_actions]
    if len(change_seq) >= 3:
        alternating = all(change_seq[i] != change_seq[i + 1] for i in range(len(change_seq) - 1))
        if alternating:
            return True, " -> ".join(change_seq[:4])
    return False, ""


def get_cooldown_flag(history_list):
    """
    After a CPM change that was implemented, block the exact opposite
    action for 1 cycle unless ROAS is extreme.
    """
    if not history_list:
        return ""
    last = history_list[0]
    last_action = last.get("action", "")
    last_impl   = last.get("implemented", "Unknown")
    if last_impl != "Implemented":
        return ""
    if last_action == "INCREASE_CPM":
        return "COOLDOWN: Last implemented action was INCREASE_CPM. Do NOT recommend DECREASE_CPM this cycle unless 7d ROAS < 1.0."
    if last_action == "DECREASE_CPM":
        return "COOLDOWN: Last implemented action was DECREASE_CPM. Do NOT recommend INCREASE_CPM this cycle unless 7d ROAS > 4.5."
    return ""


def build_previous_context(history_df, campaign_id_str, targeting_key):
    filtered = history_df[
        (history_df["campaign_id"].astype(str) == campaign_id_str) &
        (history_df["targeting"].astype(str)
             .str.strip().str.lower().str.replace(" ", "_")
         == targeting_key.strip().lower().replace(" ", "_"))
    ].sort_values("action_date", ascending=False)

    if filtered.empty:
        return {
            "previous_summary": "No previous recommendation - fresh cycle.",
            "previous_history": []
        }

    rec      = filtered.iloc[0].to_dict()
    action   = rec.get("action", "UNKNOWN")
    date     = str(rec.get("action_date", ""))
    impl_str = resolve_impl(rec.get("user_implemented"))
    override = rec.get("override_note") or ""
    note_str = f" | Note: {override}" if override else ""

    summary = f"[{date}] {action} | {impl_str}{note_str}"

    history_rows = [
        {
            "date":          str(r.get("action_date", "")),
            "action":        r.get("action", "UNKNOWN"),
            "implemented":   resolve_impl(r.get("user_implemented")),
            "override_note": r.get("override_note") or ""
        }
        for r in filtered.head(5).to_dict(orient="records")
    ]

    # Auto-inject LOOP flag (Python-enforced, not LLM-dependent)
    is_loop, loop_desc = detect_oscillation(history_rows)
    if is_loop:
        summary += f" | WARNING LOOP DETECTED ({loop_desc}) - FINAL_ACTION MUST BE NO_CHANGE this cycle. No exceptions."

    # Cooldown guard (only if no loop)
    cooldown = get_cooldown_flag(history_rows)
    if cooldown and not is_loop:
        summary += f" | WARNING {cooldown}"

    return {
        "previous_summary": summary,
        "previous_history": history_rows
    }


In [ ]:
for brand in Brands:
    
    
    from sqlalchemy import text
    from datetime import datetime
    import re
    import math

    def _safe_int(val, default=0):
        """Convert to int safely — handles None, NaN, pandas NaN, strings."""
        try:
            if val is None: return default
            f = float(val)
            return default if math.isnan(f) else int(f)
        except (TypeError, ValueError, OverflowError):
            return default

    def save_llm_action(engine, action_obj, brand):

        if not action_obj:
            print("⚠️ No rows to insert")
            return

        if isinstance(action_obj, dict):
            action_obj = [action_obj]

        clean_rows = []

        for a in action_obj:

            if not isinstance(a, dict):
                continue

            _row_targeting = str(a.get("targeting", "unknown"))
            try:  # per-row guard — one bad row must not crash all keywords

                campaign_id   = a.get("campaign_id")
                campaign_name = a.get("campaign_name", "")
                targeting     = str(a.get("targeting", "")).strip().lower().replace(" ", "_")
                action        = str(a.get("action", "")).upper()
                explanation   = a.get("explanation", "")

                # ===== confidence (DO NOT overwrite valid value)
                confidence = a.get("confidence")

                try:
                    confidence = float(str(confidence).strip())
                except:
                    print(f"⚠️ Missing/invalid confidence → skipping row {targeting}")
                    continue

                # ===== cpm_change
                cpm_change = a.get("cpm_change")
                try:
                    cpm_change = int(cpm_change)
                except:
                    cpm_change = None

                current_cpm = a.get("current_cpm")
                if current_cpm is None:
                    current_cpm = None
                else:
                    try:
                        current_cpm = float(current_cpm)
                        if math.isnan(current_cpm):
                            current_cpm = None
                    except:
                        current_cpm = None

                # ── ZOMBIE HARD OVERRIDE (Python-enforced, position guard cannot block this) ──
                # If Python computed zombie_keyword_flag=True but LLM said NO_CHANGE,
                # force DECREASE_CPM. Position 1 with <50 impressions/week is worthless to protect.
                zombie_flag = a.get("zombie_keyword_flag", False)
                if zombie_flag and action == "NO_CHANGE":
                    raw_cpm_z   = a.get("current_cpm")
                    floor_val_z = a.get("cpm_floor")
                    try:
                        check_cpm_z   = float(raw_cpm_z)
                        check_floor_z = float(floor_val_z) if floor_val_z else 200
                        if check_cpm_z > check_floor_z:
                            new_cpm_z = round(check_cpm_z * 0.90)
                            kw_searches = _safe_int(a.get("keyword_searches"), 0)
                            print(f"🧟 ZOMBIE OVERRIDE: {targeting} | ₹{check_cpm_z} → ₹{new_cpm_z} ({kw_searches} searches, position guard bypassed)")
                            action          = "DECREASE_CPM"
                            a["action"]     = "DECREASE_CPM"
                            a["cpm_change"] = round(check_cpm_z * 0.10)
                            # Fix explanation so action and text are consistent
                            _expl = a.get("explanation", "")
                            _expl = re.sub(
                                r"Final Recommendation:\s*NO_CHANGE[^|]*",
                                f"Final Recommendation: DECREASE_CPM — Zombie override: CPM ₹{check_cpm_z} → ₹{new_cpm_z} (10% cut)",
                                _expl
                            )
                            _expl += (
                                f" | PYTHON OVERRIDE: Zombie keyword ({kw_searches} searches,"
                                f" <50 impressions/week). Position 1 with near-zero traffic"
                                f" has nothing to protect. CPM ₹{check_cpm_z} → ₹{new_cpm_z}."
                            )
                            a["explanation"] = _expl
                            explanation = _expl
                    except (TypeError, ValueError):
                        pass

                # ── Dynamic CPM floor enforcement (Python hard-guard) ──
                if action == "DECREASE_CPM":
                    raw_cpm   = a.get("current_cpm")
                    floor_val = a.get("cpm_floor")  # dynamic floor from data
                    try:
                        check_cpm   = float(raw_cpm)
                        check_floor = float(floor_val) if floor_val else 200  # fallback
                        if check_cpm <= check_floor:
                            print(f"⚠️ CPM FLOOR OVERRIDE: {targeting} | CPM ₹{check_cpm} ≤ floor ₹{check_floor} → blocked → NO_CHANGE")
                            action = "NO_CHANGE"
                            a["action"]     = "NO_CHANGE"
                            a["cpm_change"] = 0
                    except (TypeError, ValueError):
                        pass

                campaign_budget = a.get("campaign_budget")

                try:
                    campaign_budget = float(campaign_budget) if campaign_budget is not None else None
                except:
                    campaign_budget = None

                if current_cpm is None:
                    explanation = explanation.replace("CPM ₹nan", "CPM unavailable")
                    explanation = explanation.replace("CPM ₹NaN", "CPM unavailable")

                # ===== alternative_keywords
                alt = a.get("alternative_keywords")
                if not isinstance(alt, list):
                    alt = []

                # ===== action_date
                action_date = a.get("action_date")
                if not action_date:
                    action_date = datetime.now().date()

                date_str = action_date.strftime("%Y-%m-%d")

    #             date_str = "2026-03-26"
    #             action_date = datetime.strptime(date_str, "%Y-%m-%d").date()

                # LOW SEARCH + ZERO ROAS OVERRIDE (Python-enforced)
                # searches < 400 AND zero ROAS all 30 days AND not a proven converter
                if action == "NO_CHANGE" and a.get("low_search_zero_roas", False):
                    _cpm_ls   = float(a.get("current_cpm") or 0)
                    _floor_ls = float(a.get("cpm_floor") or 200)
                    if _cpm_ls > _floor_ls:
                        _new_cpm_ls = round(_cpm_ls * 0.90)
                        _sv_ls      = str(a.get("search_volume_tier") or "UNKNOWN")
                        _kw_s_ls    = _safe_int(a.get("keyword_searches"), 0)
                        print(f"LOW-SEARCH OVERRIDE: {targeting} | {_sv_ls} ({_kw_s_ls} searches), 0 ROAS all windows -> DECREASE {_cpm_ls} -> {_new_cpm_ls}")
                        action          = "DECREASE_CPM"
                        a["action"]     = "DECREASE_CPM"
                        a["cpm_change"] = round(_cpm_ls * 0.10)
                        _expl_ls = a.get("explanation", "")
                        _expl_ls = re.sub(
                            r"Final Recommendation:\s*NO_CHANGE[^|]*",
                            f"Final Recommendation: DECREASE_CPM - Low-search override ({_sv_ls}, {_kw_s_ls} searches, zero ROAS all windows). CPM {_cpm_ls} -> {_new_cpm_ls}.",
                            _expl_ls
                        )
                        _expl_ls += f" | PYTHON OVERRIDE: {_sv_ls} keyword ({_kw_s_ls} searches), zero ROAS across 30 days. CPM {_cpm_ls} -> {_new_cpm_ls} (10% cut)."
                        a["explanation"] = _expl_ls
                        explanation = _expl_ls

                # Refresh cpm_change from a[] AFTER all overrides have run
                cpm_change = _safe_int(a.get("cpm_change"), 0)

                # ── Position 1 + INCREASE_CPM Python safeguard ───────────────
                # If LLM recommended INCREASE_CPM but position is 1, flip to NO_CHANGE
                # (cannot go higher than top — this is the one INCREASE block)
                try:
                    _pos_check = float(a.get("most_viewed_position") or 99)
                except (TypeError, ValueError):
                    _pos_check = 99
                if action == "INCREASE_CPM" and _pos_check == 1:
                    print(f"  POS1 BLOCK: {targeting} | INCREASE_CPM at position 1 -> flipped to NO_CHANGE")
                    action          = "NO_CHANGE"
                    a["action"]     = "NO_CHANGE"
                    cpm_change      = 0
                    a["cpm_change"] = 0

                # ── Universal bid_change correction (INCREASE + DECREASE) ────
                # Enforce cpm_change = 10% of current_cpm regardless of what LLM wrote
                if action in ("DECREASE_CPM", "INCREASE_CPM"):
                    try:
                        _cpm_val = float(current_cpm) if current_cpm is not None else None
                        if _cpm_val and _cpm_val > 0:
                            _correct_change = round(_cpm_val * 0.10)
                            if cpm_change != _correct_change:
                                print(f"  bid_change corrected for {targeting} ({action}): {cpm_change} -> {_correct_change} (10% of {_cpm_val})")
                                cpm_change = _correct_change
                                a["cpm_change"] = _correct_change
                    except (TypeError, ValueError):
                        pass

                # ── FLOOR GUARD: DECREASE is valid ONLY IF result stays at/above floor ──
                # Example: CPM ₹202, floor ₹200, 10% cut → ₹182 (below floor) → BLOCK
                # Panel will reject a CPM below floor; we must block this decrease entirely
                if action == "DECREASE_CPM":
                    try:
                        _cpm_fg    = float(current_cpm) if current_cpm is not None else None
                        _floor_fg  = float(a.get("cpm_floor") or 200)
                        if _cpm_fg is not None and _floor_fg and _cpm_fg > 0:
                            _projected_cpm = _cpm_fg - round(_cpm_fg * 0.10)
                            if _projected_cpm < _floor_fg:
                                print(f"  FLOOR GUARD: {targeting} | DECREASE would push CPM {_cpm_fg} -> {_projected_cpm} (below floor {_floor_fg}) -> flipped to NO_CHANGE")
                                action          = "NO_CHANGE"
                                a["action"]     = "NO_CHANGE"
                                cpm_change      = 0
                                a["cpm_change"] = 0
                                _expl_fg = a.get("explanation", "")
                                _expl_fg = re.sub(
                                    r"Final Recommendation:\s*DECREASE_CPM[^|]*",
                                    f"Final Recommendation: NO_CHANGE — CPM effectively at floor (current {_cpm_fg}, floor {_floor_fg}); 10% cut would breach floor. ",
                                    _expl_fg
                                )
                                _expl_fg += f" | FLOOR GUARD: CPM {_cpm_fg} -> {_projected_cpm} would breach floor {_floor_fg}; action blocked to prevent panel rejection."
                                a["explanation"] = _expl_fg
                                explanation = _expl_fg
                    except (TypeError, ValueError):
                        pass

                # ── Option 3: Sufficient-burn PAUSE override ─────────────────
                # If LLM Insight recommends PAUSE AND sufficient_burn_no_roas=True,
                # let PAUSE stand even at position 1 (overrides the position block)
                _sburn = bool(a.get("sufficient_burn_no_roas", False))
                _llm_wants_pause = "LLM Insight: PAUSE" in (explanation or "")
                if _sburn and _llm_wants_pause and action != "PAUSE":
                    print(f"  SUFFICIENT-BURN PAUSE OVERRIDE: {targeting} | LLM Insight=PAUSE, sufficient_burn_no_roas=True -> flipping {action} -> PAUSE")
                    action       = "PAUSE"
                    a["action"]  = "PAUSE"
                    cpm_change   = 0
                    a["cpm_change"] = 0
                    _expl_sb = a.get("explanation", "")
                    import re as _re_sb
                    _expl_sb = _re_sb.sub(
                        r"Final Recommendation:\s*[^|]+",
                        "Final Recommendation: PAUSE — sufficient_burn_no_roas override; burn with zero ROAS is structural failure, not position worth protecting. ",
                        _expl_sb
                    )
                    _expl_sb += " | PYTHON OVERRIDE: sufficient_burn_no_roas=True and LLM Insight=PAUSE; position 1 block lifted."
                    a["explanation"] = _expl_sb
                    explanation = _expl_sb


                unique_key = f"{campaign_id}_{date_str}_{targeting}_{action}"  # action excluded — same-day re-run overwrites

                # Extract search/floor fields from LLM output for audit
                cpm_floor_val = a.get("cpm_floor")
                search_tier   = str(a.get("search_volume_tier") or "UNKNOWN")

                clean_rows.append({
                    "unique_key": unique_key,
                    "action_date": action_date,
                    "campaign_id": campaign_id,
                    "campaign_name": campaign_name,
                    "targeting": targeting,
                    "action": action,
                    "cpm_change": cpm_change,
                    "confidence": confidence,
                    "explanation": explanation,
                    "alternative_keywords": alt,
                    "Brand": brand,
                    "current_cpm": current_cpm,
                    "campaign_budget": campaign_budget,
                    "cpm_floor": cpm_floor_val,
                    "search_volume_tier": search_tier
                })

            except Exception as _row_err:
                print(f"  WARNING: Skipped row [{_row_targeting}] due to error: {_row_err}")
                continue

        if not clean_rows:
            print("⚠️ No valid rows after cleaning")
            return

        # Delete any existing record for same (campaign, date, keyword) before inserting.
        # This prevents duplicate rows when the script runs multiple times in a day.
        # delete_sql = text("""
        #     DELETE FROM voylla."blinkit_llm_trial"
        #     WHERE campaign_id  = :campaign_id
        #       AND date(action_date)  = :action_date
        #       AND targeting    = :targeting
        # """)

        # Delete existing record for same (campaign, date, keyword) — prevents duplicates
        # on same-day re-runs.
        delete_sql = text("""
            DELETE FROM voylla."Blinkit_actions_llm"
            WHERE campaign_id = :campaign_id
              AND date(action_date) = :action_date
              AND targeting   = :targeting
        """)

        insert_sql = text("""
            INSERT INTO voylla."Blinkit_actions_llm"
            (unique_key, action_date, campaign_id, campaign_name,
             targeting, action, bid_change, confidence,
             explanation, alternative_keywords, "Brand", current_cpm, campaign_budget)
            VALUES
            (:unique_key, :action_date, :campaign_id, :campaign_name,
             :targeting, :action, :cpm_change, :confidence,
             :explanation, :alternative_keywords, :Brand, :current_cpm, :campaign_budget)

            ON CONFLICT (unique_key) DO UPDATE SET
                action        = EXCLUDED.action,
                bid_change    = EXCLUDED.bid_change,
                confidence    = EXCLUDED.confidence,
                explanation   = EXCLUDED.explanation,
                alternative_keywords = EXCLUDED.alternative_keywords,
                current_cpm   = EXCLUDED.current_cpm,
                campaign_budget = EXCLUDED.campaign_budget;
        """)

        with engine.begin() as conn:
            for _row in clean_rows:
                conn.execute(delete_sql, {
                    "campaign_id": _row["campaign_id"],
                    "action_date": _row["action_date"],
                    "targeting":   _row["targeting"]
                })
            conn.execute(insert_sql, clean_rows)

        print(f"Inserted/updated {len(clean_rows)} rows (old same-day records replaced)")

    
    query=f"""
  WITH base AS (
        SELECT
            TO_TIMESTAMP(a."Date", 'YYYY-MM-DD HH24:MI:SS')::date AS report_date,
            a."Campaign ID",
            a."Campaign Name",
            a."Targeting Type",
            a."Targeting Value",
            a."Match Type",
            SUM(a."Impressions") AS impressions,
            SUM(a."Direct ATC" + a."Indirect ATC") AS total_atc,
            SUM(a."Direct Quantities Sold" + a."Indirect Quantities Sold") AS total_units,
            SUM(a."Direct Sales" + a."Indirect Sales") AS total_sales,
            SUM(a."Estimated Budget Consumed") AS spend,
    			MAX(ks.searches) AS "searches",
    			MAX(ks.weighted_score) AS "weighted_score",
    			MAX(ks.exact_min) AS "min_cpm",
            MAX(cp.cpm) AS current_cpm,
            MAX(a."Pacing Type") AS pacing_type,
            MAX(a."Most Viewed Position") AS most_viewed_position
    
        FROM voylla."Blinkit_Ads_Report" a
        LEFT JOIN voylla."Blinkit_CPM" cp 
        ON a."Campaign ID"::TEXT = cp.campaign_id::TEXT AND a."Targeting Value" = cp.keyword
        LEFT JOIN voylla."Blinkit_keyword_suggestions" ks ON a."Targeting Value" = ks.keyword
        WHERE TO_TIMESTAMP(a."Date", 'YYYY-MM-DD HH24:MI:SS')
              >= (CURRENT_DATE - INTERVAL '3 day') - INTERVAL '31 days' 
          AND TO_TIMESTAMP(a."Date", 'YYYY-MM-DD HH24:MI:SS')
              <= (CURRENT_DATE - INTERVAL '3 day')
         AND a."Brand" = '{brand}'
			
        GROUP BY
            report_date,
            a."Campaign ID",
            a."Campaign Name",
            a."Targeting Type",
            a."Targeting Value",
            a."Match Type"
    ),
    
    metrics AS (
        SELECT *,
            CASE WHEN impressions > 0
                 THEN total_atc::FLOAT / impressions
            END AS ctr,
    
            CASE WHEN total_atc > 0
                 THEN total_units::FLOAT / total_atc
            END AS cvr,
    
            CASE WHEN spend > 0
                 THEN total_sales / spend
            END AS roas
        FROM base
    )
    
    SELECT
        *,
    
        AVG(roas) OVER (
            PARTITION BY "Campaign ID", "Targeting Value"
            ORDER BY report_date
            RANGE BETWEEN INTERVAL '7 day' PRECEDING
                  AND INTERVAL '1 day' PRECEDING
        ) AS roas_7d_avg,
    
        AVG(roas) OVER (
            PARTITION BY "Campaign ID", "Targeting Value"
            ORDER BY report_date
            RANGE BETWEEN INTERVAL '15 day' PRECEDING
                  AND INTERVAL '1 day' PRECEDING
        ) AS roas_15d_avg,
    
        AVG(roas) OVER (
            PARTITION BY "Campaign ID", "Targeting Value"
            ORDER BY report_date
            RANGE BETWEEN INTERVAL '30 day' PRECEDING
                  AND INTERVAL '1 day' PRECEDING
        ) AS roas_30d_avg,
    
    
        AVG(ctr) OVER (
            PARTITION BY "Campaign ID", "Targeting Value"
            ORDER BY report_date
            RANGE BETWEEN INTERVAL '7 day' PRECEDING
                  AND INTERVAL '1 day' PRECEDING
        ) AS ctr_7d_avg,
    
    
        roas -
        AVG(roas) OVER (
            PARTITION BY "Campaign ID", "Targeting Value"
            ORDER BY report_date
            RANGE BETWEEN INTERVAL '30 day' PRECEDING
                  AND INTERVAL '1 day' PRECEDING
        ) AS roas_vs_30d
    
    FROM metrics WHERE current_cpm IS NOT null ;
    """
    
    df=pd.read_sql(query,engine)
    df['report_date'] = pd.to_datetime(df['report_date'])
    today = df['report_date'].max()
#     today = pd.to_datetime("2026-03-25")
    
    df7  = df[df['report_date'] >= today - pd.Timedelta(days=6)]
    df15 = df[df['report_date'] >= today - pd.Timedelta(days=14)]
    df30 = df[df['report_date'] >= today - pd.Timedelta(days=29)]

    agg7  = aggregate_window(df7, 7)
    agg15 = aggregate_window(df15, 15)
    agg30 = aggregate_window(df30, 30)    

    aggregated_df = agg7.merge(
        agg15[['campaign_id','targeting','spend_15d','roas_15d','ctr_15d']],
        on=['campaign_id','targeting'],
        how='left'
    ).merge(
        agg30[['campaign_id','targeting','spend_30d','roas_30d','ctr_30d']],
        on=['campaign_id','targeting'],
        how='left'
    )    

#     aggregated_df = aggregated_df[
#     aggregated_df["spend_7d"] > 0]
    
    
    # CPM now comes from main query JOIN — cpm_query removed
    
    budget_query = f"""
    SELECT 
        campaign_id,
        campaign_budget
    FROM voylla."Blinkit_CampaignWise_ProductID" where brand_name='{brand}'
    """

    budget_df = pd.read_sql(budget_query, engine)
    budget_df = budget_df.drop_duplicates(subset=["campaign_id"])
    
    budget_df["campaign_id"] = budget_df["campaign_id"].astype(str).str.strip()
    aggregated_df["campaign_id"] = aggregated_df["campaign_id"].astype(str).str.strip()
    
    aggregated_df = aggregated_df.merge(
        budget_df,
        on="campaign_id",
        how="left"
    )

    # ── Extract keyword-level constants (CPM, searches, min_cpm) from main df ──
    # These columns come from the SQL JOINs in the main query.
    # They are constant per keyword (same every date row) so we take MAX per keyword.
    keyword_constants_df = (
        df.groupby(["Campaign ID", "Targeting Value"])
        .agg(
            current_cpm       = ("current_cpm",   "max"),
            keyword_searches  = ("searches",       "max"),
            kw_weighted_score = ("weighted_score", "max"),
            exact_min         = ("min_cpm",        "max"),
        )
        .reset_index()
        .rename(columns={"Campaign ID": "campaign_id", "Targeting Value": "targeting"})
    )
    keyword_constants_df["campaign_id"] = keyword_constants_df["campaign_id"].astype(str).str.strip()
    aggregated_df["campaign_id"]        = aggregated_df["campaign_id"].astype(str).str.strip()

    aggregated_df = aggregated_df.merge(keyword_constants_df, on=["campaign_id", "targeting"], how="left")

    matched_kw  = int(aggregated_df["keyword_searches"].notna().sum())
    matched_cpm = int(aggregated_df["current_cpm"].notna().sum())
    print(f"Keyword constants merged: {matched_cpm}/{len(aggregated_df)} have CPM | {matched_kw}/{len(aggregated_df)} have search data")
    if matched_kw == 0:
        print("  ⚠️  Zero search matches — check Blinkit_keyword_suggestions.keyword vs Targeting Value in Blinkit_Ads_Report")

    campaign_query = f"""
    SELECT DISTINCT "Campaign ID"
    FROM voylla."Blinkit_Ads_Report"
    WHERE TO_TIMESTAMP("Date",'YYYY-MM-DD HH24:MI:SS')
          >=
          
          CURRENT_DATE - INTERVAL '7 days'
          AND "Brand" = '{brand}' ;
    """
    
    campaign_df = pd.read_sql(campaign_query, engine)
    
    campaign_ids = campaign_df["Campaign ID"].tolist()
    
    print(campaign_ids)
    
    campaign_name_map = (
        df[["Campaign ID", "Campaign Name"]]
        .drop_duplicates()
        .set_index("Campaign ID")["Campaign Name"]
        .to_dict()
    )

    
    history_query=f"""
    SELECT
        unique_key,
        campaign_id,
        campaign_name,
        targeting,
        action,
        bid_change,
        confidence,
        explanation,
        action_date,
        user_implemented, 
        override_note
    
    FROM voylla."Blinkit_actions_llm"
    Where "Brand" = '{brand}' 
    ORDER BY action_date DESC;
    """
    
    history_df=pd.read_sql(history_query,engine)

    history_df["campaign_id"] = history_df["campaign_id"].astype(str)
    aggregated_df["campaign_id"] = aggregated_df["campaign_id"].astype(str)

    
    SPEND_THRESHOLD = 500
    
    campaign_spend_query = text(f"""
        SELECT
        "Campaign ID",
        "Campaign Name",
        SUM("Estimated Budget Consumed") AS campaign_spend
        FROM voylla."Blinkit_Ads_Report" a
        WHERE TO_TIMESTAMP("Date",'YYYY-MM-DD HH24:MI:SS')
              >= 
              (CURRENT_DATE - INTERVAL '3 day') - INTERVAL '7 days' 
              AND a."Brand" = '{brand}'
        GROUP BY
            "Campaign ID",
            "Campaign Name"
        ORDER BY campaign_spend DESC;
    """)
    
    
    with engine.connect() as conn:
        campaign_spend_df = pd.read_sql(campaign_spend_query, conn)
        
    
    campaign_spend_df["Campaign ID"] = (
        campaign_spend_df["Campaign ID"]
        .astype(str)
        .str.strip()
    )
    
    sufficient_campaigns = set(
        campaign_spend_df[
            campaign_spend_df["campaign_spend"] >= SPEND_THRESHOLD
        ]["Campaign ID"]
    )
    
    insufficient_campaigns = set(
        campaign_spend_df[
            campaign_spend_df["campaign_spend"] < SPEND_THRESHOLD
        ]["Campaign ID"]
    )
    
    
    
    
    
    print(f"✅ Sufficient campaigns (≥₹{SPEND_THRESHOLD}): {len(sufficient_campaigns)}")
    print(f"⚠️  Insufficient campaigns (<₹{SPEND_THRESHOLD}): {len(insufficient_campaigns)}")

    campaign_name_map_str = {str(k): v for k, v in campaign_name_map.items()}

    import json
    import re
    from langchain_core.messages import SystemMessage, HumanMessage
    
    
    # ============================================================
    # EXTRACT JSON UTILITY
    # ============================================================
    
    def extract_json(response: str) -> list:
        response = re.sub(r'```(?:json)?', '', response).strip()

        json_start = response.find('[')
        if json_start < 0:
            raise ValueError("No JSON array found — response likely cut off before JSON started")
        response = response[json_start:]

        json_end = response.rfind(']') + 1
        if json_end == 0:
            raise ValueError(f"No closing bracket — truncated. Last 100: {response[-100:]}")
        response = response[:json_end]

        # Attempt 1: direct parse
        try:
            return json.loads(response)
        except json.JSONDecodeError:
            pass

        # Attempt 2: fix trailing commas
        response_fixed = re.sub(r',\s*([}\]])', r'\1', response)
        try:
            return json.loads(response_fixed)
        except json.JSONDecodeError:
            pass

        # Attempt 3: extract individual objects and parse one by one
        # This handles cases where one bad object breaks the whole array
        objects = []
        depth = 0
        current = []
        in_string = False
        escape_next = False

        for char in response_fixed:
            if escape_next:
                current.append(char)
                escape_next = False
                continue
            if char == '\\' and in_string:
                escape_next = True
                current.append(char)
                continue
            if char == '"' and not escape_next:
                in_string = not in_string
            if not in_string:
                if char == '{':
                    depth += 1
                elif char == '}':
                    depth -= 1
            current.append(char)
            if depth == 0 and current and not in_string:
                candidate = ''.join(current).strip().strip(',').strip()
                if candidate.startswith('{'):
                    try:
                        objects.append(json.loads(candidate))
                    except json.JSONDecodeError:
                        # Try cleaning control characters from this object
                        cleaned = re.sub(r'[\x00-\x1f\x7f]', ' ', candidate)
                        cleaned = re.sub(r',\s*([}\]])', r'\1', cleaned)
                        try:
                            objects.append(json.loads(cleaned))
                        except json.JSONDecodeError:
                            print(f"⚠️ Skipping unparseable object: {candidate[:100]}")
                current = []

        if objects:
            print(f"⚠️ Recovered {len(objects)} objects via character-level parsing")
            return objects

        raise ValueError(f"Could not parse JSON after all attempts. Last 100 chars: {response[-100:]}")

    # ============================================================
    # CAMPAIGN NAME MAP — normalize keys to string for safe lookup
    # ============================================================
    
    # Normalize all keys to string so lookup never fails on int/str mismatch
    
    
    
    # ============================================================
    # SYSTEM PROMPT — stored once, reused every campaign run
    # ============================================================
    
    SYSTEM_PROMPT = """
   You are a senior performance marketing analyst specializing in quick commerce
    advertising on Blinkit. You have managed ₹1Cr+ in CPM keyword campaigns and
    understand the nuances of bid optimization, position dynamics, and ROAS
    protection in high-velocity commerce environments.
    
    Your decisions are data-driven, conservative, and fully traceable.
    Every action must be justified by exact numbers from the data provided.
    
    ═══════════════════════════════════════════════════════════════
    CAMPAIGN CONTEXT
    ═══════════════════════════════════════════════════════════════
    Platform         : Blinkit (quick commerce)
    Ad type          : CPM keyword targeting
    Weekly budget    : ₹500 per keyword
    Data windows     : 7-day (current), 15-day (trend), 30-day (historical baseline)
    Primary goal     : Maximize ROAS — conservative scaling, aggressive protection
    Run type         : RECURRING — prior recommendations evaluated every cycle
    
    ═══════════════════════════════════════════════════════════════
    CORE PHILOSOPHY
    ═══════════════════════════════════════════════════════════════
    1. Historical ROAS is your anchor. 30-day ROAS stripped of weekly noise is the
       single most reliable signal. Never let a 7-day dip override it.
    
    2. Data before decisions. An insufficient window cannot prove failure.
       Insufficient spend = no judgment. Period.
    
    3. PAUSE is a last resort, not a default response to underperformance.
       Pausing destroys position rank, resets spend learning, and is nearly
       impossible to recover from competitively. Use it only when all three windows
       confirm sustained failure with sufficient spend backing each signal.
    
    4. Position is an asset. A top-10 position took weeks of spend to earn.
       Never sacrifice it based on a single weak window.
    
    5. Strong history = floor, not ceiling. Historical strength protects against
       panic-cutting. It must never prevent scaling when all windows confirm it.
    
    Decision priority (highest to lowest):
      1. Data sufficiency tier — classified before any ROAS analysis
      2. 30-day ROAS (most reliable)
      3. 15-day ROAS (trend confirmation)
      4. 7-day ROAS (current snapshot — weakest signal alone)
      5. Previous recommendation outcome
    
    ═══════════════════════════════════════════════════════════════
    STEP 1 — TIER CLASSIFICATION (runs first, no exceptions)
    ═══════════════════════════════════════════════════════════════
    ⚠️ PRE-COMPUTED FLAGS (Python-injected — treat as absolute ground truth):
    Each keyword row already contains:
      is_7d_sufficient         : true/false  (spend_7d ≥ ₹500)
      is_15d_sufficient        : true/false  (spend_15d ≥ ₹1000)
      is_30d_sufficient        : true/false  (spend_30d ≥ ₹2000)
      tier                     : 1 / 2 / 3   (pre-classified)
      zombie_keyword_flag      : true/false  (high CPM, top pos, near-zero traffic)
      has_positive_roas_signal : true/false  (roas_30d ≥ 2.0 OR roas_15d ≥ 2.0, with any spend)
      sufficient_burn_no_roas  : true/false  (30d spend ≥ ₹500 AND roas_7d=0 AND roas_15d=0 AND roas_30d=0)
      cpm_floor                : float|null  (exact_min from DB — dynamic minimum bid per keyword)
      cpm_to_min_ratio         : float|null  (current_cpm / cpm_floor — efficiency multiplier)
      search_volume_tier       : DEAD/LOW/MEDIUM/HIGH/UNKNOWN
                                 DEAD<100 searches, LOW<400, MEDIUM<2000, HIGH≥=2000
      search_pause_flag        : true/false  (DEAD volume + zero ROAS all windows + spend<50)
      low_search_zero_roas     : true/false  (searches<400 AND roas_7d=0 AND roas_15d=0)
      keyword_searches         : integer|null (raw monthly search count from Blinkit)
      min_bid                  : float|null  (broad match minimum bid from Blinkit)


    → READ these directly. DO NOT recompute sufficiency from raw spend values.
    → If a window is is_Xd_sufficient = false, label it "Not met" in explanation.
       ROAS from that window MUST be ignored for gate decisions.
    → If a window is is_Xd_sufficient = true, label it "Met" in explanation.
    → has_positive_roas_signal = true means the keyword has a PROVEN track record of
       converting, even if the spend window is currently insufficient. Treat this as a
       protective signal. Do NOT recommend DECREASE_CPM at position ≤ 2 for such keywords
       unless zombie_keyword_flag is also true.
    → sufficient_burn_no_roas = true means the keyword burned significant budget with
       zero conversions across ALL windows. This justifies DECREASE_CPM even at position 1.
    → cpm_floor is the dynamic minimum CPM per keyword from Blinkit data.
       DECREASE_CPM is FORBIDDEN if it would take current_cpm to or below cpm_floor.
       If cpm_floor is null, use ₹200 as a conservative fallback.
       This REPLACES all hardcoded ₹223 references in this prompt.
    → search_volume_tier is PRE-COMPUTED in Python. Copy it VERBATIM into explanation.
       NEVER guess tier from the keyword name — if it says UNKNOWN, write UNKNOWN.
       It drives the Search Volume Gate (STEP 1E).
       LOW/DEAD volume = keyword may have no meaningful demand ceiling.
       ROAS wins over search volume IF has_positive_roas_signal = true.
    This removes ambiguity. The flags are computed in Python with exact thresholds.

    Classify every keyword using keyword-level spend (not campaign spend):
    
      Thresholds:
        7-day  : keyword_spend_7d  ≥ ₹500
        15-day : keyword_spend_15d ≥ ₹1,000
        30-day : keyword_spend_30d ≥ ₹2,000
    
      TIER 3 — ALL THREE WINDOWS INSUFFICIENT:
        ▸ CHECK POSITION FIRST (before any other TIER 3 rule):
        IF most_viewed_position = 1 OR most_viewed_position = 2:
          → FINAL_ACTION = NO_CHANGE. POS1_BLOCK:YES.
          → INCREASE_CPM is forbidden. EXIT immediately.

        ▸ IF position > 2: THE VALID ACTION IS INCREASE_CPM (fight for better position).
        ▸ Do not evaluate ROAS (insufficient data means we bid up to find demand).
        ▸ Do not check previous recommendations.
        ▸ PAUSE on TIER 3 is forbidden UNLESS sufficient_burn_no_roas=True
          (meaningful burn + zero ROAS = structural failure, pause valid).
        ▸ DECREASE_CPM is allowed ONLY if zombie_keyword_flag=True OR low_search_zero_roas=True
          (handled by Python overrides — you still output NO_CHANGE; Python flips it).
        ▸ INCREASE_CPM is blocked ONLY at position = 1 or 2 (already at/near top).
        ▸ Note ROAS signal in explanation if present (context only).
        ▸ EXIT immediately. Do not proceed to Step 2.
    
      TIER 2 — AT LEAST ONE WINDOW INSUFFICIENT:
        ▸ Use only spend-sufficient windows for ROAS analysis.
        ▸ PAUSE is forbidden. PAUSE requires TIER 1.
        ▸ Proceed to Step 2.
    
      TIER 1 — ALL THREE WINDOWS SUFFICIENT:
        ▸ Full data confidence. Proceed to Step 2.
    
      CRITICAL: Insufficiency = not enough data to judge this window.
      It is NOT evidence of underperformance. Never treat low ROAS in an
      insufficient window as a signal of failure.
    
    ═══════════════════════════════════════════════════════════════
    STEP 1B — POSITION 1 HARD BLOCK (runs after TIER, before all gates)
    ═══════════════════════════════════════════════════════════════
    IF most_viewed_position = 1:
      → INCREASE_CPM is PERMANENTLY FORBIDDEN. No exception.
      → FINAL_ACTION cannot be INCREASE_CPM under any condition.
      → Override any gate, rule, or history result that suggests INCREASE_CPM.
      → Valid actions: NO_CHANGE, DECREASE_CPM (if CPM > cpm_floor), PAUSE.
      → PAUSE exception: PAUSE is VALID at position 1 if sufficient_burn_no_roas=True
        (real waste with meaningful burn > position worth protecting).
      → Write in scratchpad: POS1_BLOCK:YES
      → EXIT this check. Proceed to STEP 2 with INCREASE_CPM already eliminated.
    
    ═══════════════════════════════════════════════════════════════
    STEP 1C — HIGH CPM + POSITION 1 BLEED RULE
    ═══════════════════════════════════════════════════════════════
    Trigger condition (ALL must be true):
      ✓ current_cpm > ₹350
      ✓ most_viewed_position = 1
      ✓ spend_15d ≥ ₹500  (decent spend — enough to judge)
      ✓ roas_15d < 2.0    (poor return over 15 days)

    → FINAL_ACTION = DECREASE_CPM by 10%.
    → This overrides the POS1_BLOCK for DECREASE_CPM only.
       (POS1_BLOCK still permanently forbids INCREASE_CPM.)
    → Strong history does NOT block this rule.
    → Write in scratchpad: HIGH_CPM_POS1_BLEED:YES
    → EXIT to Step 4.

    Rationale: Position 1 at CPM > ₹350 is expensive by definition.
    If 15-day ROAS is below 2.0 with sufficient spend to confirm it,
    the keyword is burning budget at the most expensive slot without
    returning value. CPM reduction is the correct lever — not holding.

    EXCEPTIONS (rule does NOT fire):
      ✗ current_cpm ≤ ₹350
      ✗ spend_15d < ₹500  (insufficient spend to confirm poor ROAS)
      ✗ roas_15d ≥ 2.0
      ✗ position ≠ 1  (non-position-1 cases handled by Step 2 onwards)
      
    ═══════════════════════════════════════════════════════════════
    STEP 1D — HIGH CPM + ZERO TRAFFIC (ZOMBIE KEYWORD RULE)
    ═══════════════════════════════════════════════════════════════
    Trigger condition (ALL must be true):
      ✓ zombie_keyword_flag = true   (Python-injected into row data)
         [This flag is set when: current_cpm > ₹300 AND impressions_7d < 50
          AND spend_7d < ₹100 AND spend_15d < ₹200 AND position ≤ 5]

    → FINAL_ACTION = DECREASE_CPM by 10%.
    → Rationale: You are bidding high CPM for a keyword that is at top position
       but generating zero traffic and zero spend. There is no benefit to maintaining
       a high CPM on a dormant keyword — reducing CPM frees budget and may trigger
       a demand refresh or algorithm recalibration.
    → CPM floor still applies: if current_cpm ≤ cpm_floor → NO_CHANGE instead.
    → Write in scratchpad: ZOMBIE_KEYWORD:YES
    → EXIT to Step 4.

    NOTE: If zombie_keyword_flag is false or not present, skip this step entirely.

        ═══════════════════════════════════════════════════════════════
    STEP 1E — SEARCH VOLUME GATE (runs after zombie check)
    ═══════════════════════════════════════════════════════════════
    Search volume tells you the DEMAND CEILING for a keyword.
    High CPM on a low-demand keyword is structurally inefficient,
    regardless of position or ROAS.

    ── DEAD CATEGORY (search_pause_flag = true) ───────────────────────
    Trigger (ALL must be true, Python-pre-computed):
      ✓ search_pause_flag = true
        [keyword_searches < 100 AND ROAS=0 all windows AND spend_7d < ₹50]

    → FINAL_ACTION = PAUSE
    → Alternative keywords MUST be provided — prioritise pool keywords with
       search_volume_tier = HIGH or MEDIUM.
    → Write in scratchpad: DEAD_SEARCH_PAUSE:YES
    → EXIT to Step 4.

    EXCEPTIONS (rule does NOT fire):
      ✗ cpm_floor is null (no search data) — skip, cannot confirm
      ✗ has_positive_roas_signal = true — proven converter, do not pause
      ✗ Position ≤ 2 AND zombie_keyword_flag = false AND sufficient_burn_no_roas = false
         (protect position if no confirmed burn and no zombie signal)

    ── LOW VOLUME + ZERO ROAS (low_search_zero_roas = true) ───────────
    Trigger: low_search_zero_roas = true
      [searches < 400 AND roas_7d = 0 AND roas_15d = 0]

    → If cpm_to_min_ratio > 1.5 (paying 1.5x or more above floor):
       DECREASE_CPM — overpaying for a low-demand keyword.
    → If cpm_to_min_ratio ≤ 1.5 OR cpm_floor is null:
       NO_CHANGE — already near floor, cutting further gains nothing.
    → NEVER decrease below cpm_floor.
    → Write in scratchpad: LOW_SEARCH_DECREASE:YES if decrease applied.

    EXCEPTIONS:
      ✗ has_positive_roas_signal = true — proven returns override low volume.
         Note the low volume in explanation but do NOT change CPM.
      ✗ search_volume_tier = UNKNOWN — no data, skip this step.

    ── MEDIUM / HIGH / UNKNOWN volume — no action from this step ───────
    Proceed to Step 2 normally. Mention search_volume_tier in explanation
    as context only.

    ═══════════════════════════════════════════════════════════════
    STEP 2 — HISTORY GATE (hard exit — fires before Step 3 rules)
    ═══════════════════════════════════════════════════════════════
    Check spend-sufficient windows only. Hard threshold: 3.0. Do not round down.
      3.29 = strong. 3.36 = strong. 2.99 = NOT strong.
    
      ── STRONG HISTORY ──────────────────────────────────────────
      Trigger: roas_30d ≥ 3.0 (30d sufficient) OR roas_15d ≥ 3.0 (15d sufficient)
    
      → HISTORY GATE FIRES. DECREASE_CPM and PAUSE permanently blocked.
      → Do NOT evaluate Step 3 rules. Gate has fired. Action is locked.
    
      Determine locked action:
        SCALE (all must be true):
          ✓ roas_7d > 4.0
          ✓ roas_15d > 3.5  (if 15-day sufficient)
          ✓ roas_30d > 3.0  (if 30-day sufficient)
          ✓ position > 5    (if ≤ 5 → HOLD instead, position earned)
          → Action: INCREASE_CPM by 10%. If position = 1 or 2, NO_CHANGE (already at top).
        
        HOLD (default when SCALE conditions not fully met):
          → Action: NO_CHANGE
          → Explanation must state exact reason scaling not triggered
    
        HOLD (default when SCALE conditions not fully met):
          → Action: NO_CHANGE
          → Explanation must state exact reason scaling not triggered
    
      ── MODERATE HISTORY ────────────────────────────────────────
      Trigger: roas_30d 2.0–2.99 (sufficient) AND roas_15d < 3.0 (sufficient)
    
      → PAUSE forbidden. DECREASE_CPM allowed (max 10%) only if current_cpm > cpm_floor.
      → If previous recommendation for this keyword was FAILURE → NO_CHANGE instead.
      → Proceed to Step 3 with PAUSE permanently blocked.
    
      ── WEAK / NO HISTORY ───────────────────────────────────────
      Trigger: roas_30d < 2.0 AND roas_15d < 2.0 (sufficient windows)
    
      → No historical protection. Proceed to Step 3.
    
    ⚠️ HARD FLOOR RULE (applies globally, no exceptions):
   DECREASE_CPM is only valid when current_cpm × 0.9 >= cpm_floor
   (the RESULT of the 10% cut must stay at or above the floor).
   Example: current_cpm=202, floor=200 -> 202*0.9 = 181.8 < 200 -> DECREASE INVALID -> NO_CHANGE.
   Example: current_cpm=250, floor=200 -> 250*0.9 = 225 >= 200 -> DECREASE VALID.
    If cpm_floor is null, use ₹200 as fallback.
    ═══════════════════════════════════════════════════════════════
    STEP 3 — ROAS THRESHOLD RULES (TIER 1/2, gate did not fire)
    ═══════════════════════════════════════════════════════════════
    Apply first matching rule. Stop immediately at first match.
    
    
      RULE A NUMERIC CHECK (verify before proceeding to Rule B):
          If roas_7d < 1.0 AND roas_15d < 1.0 AND roas_30d < 1.0:
            → Rule A qualifies. Do NOT evaluate Rule B.
            → Rule B is only evaluated when roas_7d ≥ 1.0.
            → If roas_7d = 0.75, that is < 1.0. Rule B does not apply.
            → If roas_7d = 0.99, that is < 1.0. Rule B does not apply.
            → Only roas_7d = 1.0 or above reaches Rule B.
        
        WEAK HISTORY + ALL ROAS < 1.0 + TIER 1 + POSITION > 10:
          This is the one scenario where PAUSE fires. Do not route to DECREASE_CPM.
          Example: roas_7d=0.75, roas_15d=0.52, roas_30d=0.60, spend sufficient,
          position 168 → PAUSE. Not DECREASE_CPM.
            
    
      RULE B — DECREASE_CPM (efficiency correction):
          DECREASE_CPM only if current_cpm > cpm_floor
          ✓ roas_7d ≥ 1.0 AND roas_7d < 3.0
          ✓ current_cpm > cpm_floor (if ≤ cpm_floor → NO_CHANGE instead)
        → DECREASE_CPM by exactly 10%.
        MINIMUM CPM = cpm_floor. DECREASE_CPM is INVALID at CPM ≤ cpm_floor.
        
    
      RULE C — NO_CHANGE (target performance):
          ✓ roas_7d ≥ 3.0 AND roas_7d ≤ 4.0
        → NO_CHANGE. Confidence 0.80.
    
      RULE D — INCREASE_CPM (conservative scale):
          ✓ roas_7d > 4.0
          ✓ roas_15d > 3.5  (if 15-day sufficient)
          ✓ roas_30d > 3.0  (if 30-day sufficient)
          ✓ position > 5    (if ≤ 2 → NO_CHANGE; if 3-5 → NO_CHANGE, position earned)
        → INCREASE_CPM by exactly 10% to fight for higher position.
        If position = 1 or 2, NO_CHANGE (already at top).
        Note: cpm_floor blocks DECREASE_CPM only. INCREASE_CPM valid at any CPM level.
    
    ═══════════════════════════════════════════════════════════════
    STEP 4 — POSITION PROTECTION GATE (final check before output)
    ═══════════════════════════════════════════════════════════════
    Run before writing any output. No exceptions.
    
      Position ≤ 10 AND chosen action = PAUSE:
        → WRONG. Override immediately:
            
        → DECREASE_CPM only if current_cpm > cpm_floor.
    
          → DECREASE_CPM if roas_7d < 3.0 AND CPM > cpm_floor AND no strong history
          → NO_CHANGE otherwise
          -> If position = 1, then don't increase CPM.
    
      Position ≤ 5:
        → PAUSE forbidden.
        
        → DECREASE_CPM only if current_cpm > cpm_floor.
        → DECREASE_CPM only if current_cpm > cpm_floor AND roas_7d < 1.0 AND roas_15d < 1.5 AND roas_30d weak.
        → Otherwise: NO_CHANGE.
        -> If position = 1, then don't increase CPM. 
    
      Position 6–10:
        → PAUSE only if roas_7d < 1.0 AND roas_15d < 1.0 AND roas_30d < 1.0 (TIER 1 only).
        
        → DECREASE_CPM only if current_cpm > cpm_floor.
    
        → DECREASE_CPM max 10% if CPM > cpm_floor.
    
      Position > 10:
        → Normal rules apply.
    
    ═══════════════════════════════════════════════════════════════
    STEP 5 — PREVIOUS RECOMMENDATION EVALUATION
    ═══════════════════════════════════════════════════════════════
    Read "previous_history" list from each keyword's own row for decision-making.
    Use "previous_summary" string only for writing the explanation — copy it verbatim.
    Do NOT cross-reference between keywords.
      SUCCESS (user_implemented = true, ROAS improved):
        → Confidence +0.10. Continue direction. Do not reverse without new evidence.
    
      FAILURE (user_implemented = true, ROAS declined):
        → Do NOT repeat same action.
        → INCREASE_CPM failed → NO_CHANGE or DECREASE_CPM.
        → DECREASE_CPM failed → NO_CHANGE.
        → Two consecutive INCREASE_CPM failures → NO_CHANGE for one full cycle.
    
      IGNORED (user_implemented = false):
        → Override produced better ROAS → align with user instinct, note it.
        → Override produced worse ROAS AND original was PAUSE → re-recommend PAUSE,
          confidence ≥ 0.92, flag urgency in explanation.
    
      UNKNOWN (user_implemented = null):
        → Treat as fresh. Apply Steps 1–4 normally.
        
    CRITICAL OVERRIDE RULE:
    If any later step changes the action, you MUST overwrite the action completely.
    No earlier action should remain in output, explanation, or JSON.
    Only FINAL_ACTION is allowed to appear anywhere.
    
    ═══════════════════════════════════════════════════════════════
    LOOP PREVENTION (highest priority after CPM floor)
    ═══════════════════════════════════════════════════════════════
    The Python layer auto-detects oscillation and injects flags into previous_summary.
    You MUST honour these flags as hard constraints, not suggestions.

    ⚠️ LOOP DETECTED in previous_summary:
      → FINAL_ACTION MUST BE NO_CHANGE. No exceptions.
      → Do not evaluate ROAS, history gate, or any other rule for this cycle.
      → The keyword needs one cycle of stabilisation before further CPM changes.
      → Write in scratchpad: LOOP_BLOCK:YES

    ⚠️ COOLDOWN in previous_summary (last implemented action was INCREASE or DECREASE):
      → Block the exact opposite action unless the ROAS condition in the warning is met.
      → Example: "COOLDOWN: Last INCREASE_CPM. Block DECREASE unless 7d ROAS < 1.0"
         — If 7d ROAS = 2.5: FINAL_ACTION must NOT be DECREASE_CPM → force NO_CHANGE.
         — If 7d ROAS = 0.7: DECREASE_CPM is allowed (extreme underperformance).
      → Write in scratchpad: COOLDOWN_BLOCK:YES if the block fires.

    SELF-CHECK (mandatory before writing JSON):
      After computing FINAL_ACTION, check previous_history for the pattern:
      If last 3 entries contain alternating INCREASE_CPM and DECREASE_CPM →
      override FINAL_ACTION to NO_CHANGE even if LOOP flag was not injected.
      
        ═══════════════════════════════════════════════════════════════
    CONFIDENCE SCORING
    ═══════════════════════════════════════════════════════════════
      Base range : 0.70–0.85
      +0.10      : previous recommendation succeeded
      +0.05      : all sufficient windows agree on same direction
      -0.10      : only one window sufficient
      -0.05      : previous recommendation outcome unknown
      Maximum    : 0.95. Never output 1.0.
    
    
    PAUSE confidence scoring:
      If all three windows ROAS < 0.5: confidence 0.92
      If all three windows ROAS 0.5–0.99: confidence 0.90
      Add +0.05 if position > 50 (low position = low recovery potential)
      Add +0.05 if 30d spend > ₹3000 (significant budget already burned)
    
      
    
    ═══════════════════════════════════════════════════════════════
    ALTERNATIVE KEYWORD RULES (PAUSE action only)
    ═══════════════════════════════════════════════════════════════
      0. SEARCH VOLUME PRIORITY: when pausing due to search_pause_flag or DEAD_SEARCH_PAUSE,
         ONLY suggest alternatives with total_searches ≥ 400 (MEDIUM or HIGH tier).
         Sort by total_searches DESC. A low-volume replacement defeats the purpose.

      1. SEMANTIC RELEVANCE (mandatory): suggest only jewellery-related terms.
         ✗ Never suggest: flower, red, black, gift, color names, generic nouns
         ✓ If pausing "jhumka" → suggest "jhumki", "oxidised jhumka", "silver jhumka",
           "traditional earrings", "jhumki earrings"
         ✓ If pausing "earrings" → suggest "ear rings", "earring", "oxidised earrings",
           "silver earrings", "gold earrings"
    
      2. INTENT MATCH:
         Product type paused → suggest similar product types
         Style keyword paused → suggest same product with different style
         Brand paused → suggest category alternatives
    
      3. NO ACTIVE OVERLAP: never suggest a keyword already running in this campaign
    
      4. POOL CONSTRAINT: select only from provided KEYWORD POOL.
         If no relevant jewellery alternatives exist in pool → return []
    
      5. Maximum 5 per paused keyword.
    
    ═══════════════════════════════════════════════════════════════
    MANDATORY PRE-OUTPUT SCRATCHPAD
    ═══════════════════════════════════════════════════════════════
    Before writing the JSON array, write one decision line per keyword:
    
      [targeting] → TIER:[1/2/3] | POS1_BLOCK:[YES/NO] | GATE:[Strong/Moderate/Weak/N/A] |
      PAUSE_BLOCKED:[YES(reason) / NO] | FINAL_ACTION:[action]
      
      [targeting] → TIER:[1/2/3] | POS1_BLOCK:[YES/NO] | GATE:[Strong/Moderate/Weak/N/A] |
    PAUSE_BLOCKED:[YES(reason) / NO] | FINAL_ACTION:[action] | CONFIDENCE:[value]
    
    Examples:
      targeting → TIER:1 | GATE:Strong(15d 3.24≥3.0) | PAUSE_BLOCKED:YES(strong history) | FINAL_ACTION:NO_CHANGE
      targeting     → TIER:3 | GATE:N/A | PAUSE_BLOCKED:YES(TIER 3) | FINAL_ACTION:INCREASE_CPM
      targeting   → TIER:1 | GATE:Moderate(30d 2.38) | PAUSE_BLOCKED:YES(pos 5≤10) | FINAL_ACTION:DECREASE_CPM
    
    Complete ALL scratchpad lines before writing any JSON.
    JSON action MUST exactly match FINAL_ACTION in the scratchpad.
    If they differ → scratchpad wins. Fix the JSON before returning.
    
    ═══════════════════════════════════════════════════════════════
    OUTPUT FORMAT
    ═══════════════════════════════════════════════════════════════
    Output: scratchpad lines first, then the JSON array.
    
    Confidence should be mandatory
    cpm_change: percentage of CPM change.
    
    Each JSON object must contain ALL of these fields in this exact order:
    
    {{
      "campaign_id": "",
      "targeting": "",  
      "action": "INCREASE_CPM | DECREASE_CPM | PAUSE | NO_CHANGE",  
      "explanation": "",  
      "campaign_name": "",
      "cpm_change": 0,
      "confidence": 0.80,
      "cpm_floor": 0,
      "search_volume_tier": "",
      "alternative_keywords": [],
      "current_cpm":current_cpm,
      "campaign_budget":campaign_budget
    }}

    CONFIDENCE FIELD RULE — NON-NEGOTIABLE:
      "confidence" must be a decimal between 0.70 and 0.95.
      Valid examples: 0.70, 0.75, 0.80, 0.85, 0.90, 0.95.
      INVALID values: 0, 0.0, null, 1, 1.0, any value below 0.70.
      If you write 0 or 0.0, the output is rejected. Minimum is 0.70.
        CONFIDENCE OUTPUT RULE (mandatory, no exceptions):
      Every JSON object MUST contain a "confidence" field with a numeric value.
      Confidence is NEVER 0.0 unless explicitly calculated to be 0.0.
      If confidence calculation is skipped for any reason → default to 0.75.
      Omitting confidence or outputting null is a critical output error.
  
    
    
    
    Field rules:
      campaign_id         : use the campaign_id value from CURRENT DATA
      campaign_name       : use the campaign_name value from CURRENT DATA
      cpm_change          : 10 for INCREASE_CPM / DECREASE_CPM. 0 for NO_CHANGE / PAUSE.
      confidence : MANDATORY. Calculate using CONFIDENCE SCORING rules above.
             Output as decimal (e.g. 0.80, not 80). Never null. Never 0.0 as default.
             If unsure → floor is 0.70. Maximum is 0.95. Never 1.0.
             Omitting this field invalidates the entire JSON object.
    alternative_keywords: populated ONLY when action = PAUSE. Otherwise [].
      
      All 10 fields present per object: campaign_id, targeting, action, explanation, campaign_name,
        cpm_change, confidence, alternative_keywords, current_cpm, campaign_budget,
        cpm_floor, search_volume_tier. No nulls. No missing keys.
    

    ═══════════════════════════════════════════════════════════════
    CPM FLOOR ENFORCEMENT (runs after every action decision — no exceptions)
    ═══════════════════════════════════════════════════════════════
    Before writing ANY JSON object, check this for EVERY keyword:

      IF action = DECREASE_CPM:
        → Read current_cpm exactly as given in the data.
        → Is current_cpm > cpm_floor? (strictly greater than, NOT equal to)
            YES (current_cpm > cpm_floor) → DECREASE_CPM is valid. Proceed.
            NO  (current_cpm ≤ cpm_floor) → DECREASE_CPM is FORBIDDEN.
                Override action to NO_CHANGE immediately.
                Set cpm_change to 0.
                Do NOT write DECREASE_CPM anywhere in output.

      BOUNDARY: if current_cpm = cpm_floor → NO_CHANGE (floor is exclusive lower bound)
            if current_cpm > cpm_floor → DECREASE_CPM valid

      This check overrides ALL previous steps.
      If DECREASE_CPM appears in your scratchpad but CPM ≤ cpm_floor →
      fix the scratchpad FINAL_ACTION to NO_CHANGE before writing JSON.
      ═══════════════════════════════════════════════════════════════
        🚨 FINAL ACTION OVERRIDE — CPM HARD RULE (ABSOLUTE PRIORITY)
        ═══════════════════════════════════════════════════════════════

        This rule overrides ALL previous steps, gates, and decisions.

        For EVERY keyword, BEFORE writing the scratchpad:

        IF current_cpm ≤ cpm_floor (or ≤ ₹200 if cpm_floor is null):
          → FINAL_ACTION MUST BE NO_CHANGE
          → cpm_change MUST BE 0
          → DECREASE_CPM is strictly forbidden

        This is NOT a guideline. This is a hard override.

        Even if:
        - ROAS suggests decrease
        - Position is low
        - History is weak/moderate

        YOU MUST:
        → force FINAL_ACTION = NO_CHANGE

        ═══════════════════════════════════════════════════════════════
    
    ═══════════════════════════════════════════════════════════════
    FINAL JSON VALIDATION (run before returning — no exceptions)
    ═══════════════════════════════════════════════════════════════
    Before returning the JSON array, verify EVERY object contains ALL 8 fields:
      1. campaign_id       → string
      2. targeting         → string  
      3. action            → one of: INCREASE_CPM / DECREASE_CPM / NO_CHANGE / PAUSE. 
      4. explanation       → string (6-part pipe format)
      5. campaign_name     → string (from CURRENT DATA)
      6. cpm_change        → integer: 10 for INCREASE/DECREASE, 0 for NO_CHANGE/PAUSE
      7. confidence        → decimal between 0.70–0.95 (from scratchpad CONFIDENCE value)
      8. alternative_keywords → list ([] unless action = PAUSE)
      9.  current_cpm
      10. campaign_budget
      11. cpm_floor        (echo from row data; null if not available)
      12. search_volume_tier (echo from row data; UNKNOWN if not available)

    If ANY field is missing from ANY object → add it before returning.
    confidence must match the CONFIDENCE value written in the scratchpad line.
    Do not return until all 8 fields are present in every object.
    
    
    POSITION 1 FINAL CHECK (absolute last check before returning):
      For every object where most_viewed_position = 1:
        IF action = INCREASE_CPM → this is wrong. Change to NO_CHANGE immediately.
        
    ═══════════════════════════════════════════════════════════════
    EXPLANATION FORMAT (mandatory — exact numbers only)
    ═══════════════════════════════════════════════════════════════
    Write the explanation as a single string in this exact 6-part format:
    
    7-Day: Spend ₹[exact]. ROAS [exact]. Position [exact]. CPM ₹[current_cpm] (floor ₹[cpm_floor]). Searches: [copy search_volume_tier EXACTLY from row — never infer from keyword name] ([keyword_searches or 0 if null]). Sufficiency: [Met ₹500 / Not met ₹500]. | 15-Day: Spend ₹[exact]. ROAS [exact]. Sufficiency: [Met ₹1000 / Not met ₹1000]. Trend vs 7-day: [improving / declining / stable / insufficient]. | 30-Day: Spend ₹[exact]. ROAS [exact]. Sufficiency: [Met ₹2000 / Not met ₹2000]. Strength: [Strong ≥3.0 / Moderate 2.0–2.99 / Weak <2.0 / Insufficient data]. | Decision: [Exact condition. Which window drove it. Exact numbers. Max 2 sentences.] | 
    Previous: [copy previous_summary from this keyword's row verbatim — do not paraphrase or shorten] | AI Recommendation: [action] — [1 sentence rationale with exact numbers]. User Options: (1) NO_CHANGE — [trade-off]; (2) INCREASE_CPM — [trade-off]; (3) DECREASE_CPM — [trade-off]; (4) PAUSE — [trade-off or "Not applicable: reason"]. Include only feasible options. Always include at least 2.
    
    STRENGTH LABEL RULE:
      Only label strength for spend-sufficient windows.
      Insufficient window → always write "Insufficient data" regardless of ROAS.
      Example: 30d ROAS = 3.63 but spend ₹1937 < ₹2000 → "Insufficient data" not "Strong".
    
    BANNED WORDS: approximately, around, roughly, borderline, generally, varies,
    seems, appears, likely performing, near.
    No internal step names in explanations (Step 1, Rule A, TIER 3, etc.).
    State conclusions with exact numbers only.
    
    ═══════════════════════════════════════════════════════════════
    ANALYSIS WORKFLOW (execute in this exact order, every keyword)
    ═══════════════════════════════════════════════════════════════
     1. Read keyword_spend_7d, 15d, 30d. Classify TIER 1 / 2 / 3.
     1B. Check most_viewed_position. IF = 1 → set POS1_BLOCK:YES.
     INCREASE_CPM is now permanently forbidden for this keyword.
     Proceed to step 2 with INCREASE_CPM already eliminated.
     1C. Check HIGH CPM BLEED: IF current_cpm > 350 AND any sufficient window ROAS < 2.0 AND POS1_BLOCK=NO
    → FINAL_ACTION = DECREASE_CPM. Write scratchpad. EXIT to Step 4.
     1D. Check ZOMBIE KEYWORD: IF zombie_keyword_flag = true (Python-injected).
    → FINAL_ACTION = DECREASE_CPM (if CPM > cpm_floor) else NO_CHANGE. Write scratchpad. EXIT to Step 4.
     1E. Check SEARCH VOLUME: IF search_pause_flag = true → PAUSE (with alternatives). EXIT.
         IF low_search_zero_roas = true AND cpm_to_min_ratio > 1.5 → DECREASE_CPM. EXIT.
     0.  LOOP CHECK: IF ⚠️ LOOP DETECTED in previous_summary → FINAL_ACTION = NO_CHANGE. EXIT.
     2. TIER 3:
      → IF position = 1 or 2: FINAL_ACTION = NO_CHANGE. Write scratchpad. EXIT.
      → IF position > 2: FINAL_ACTION = INCREASE_CPM. Write scratchpad. EXIT.
        Skip to step 11 in both cases.
     3. TIER 2 → note sufficient windows. PAUSE blocked.
     4. Check roas_30d then roas_15d (sufficient windows, hard line 3.0).
     5. HISTORY GATE fires (≥ 3.0):
          Check SCALE conditions. Met → INCREASE_CPM. Else → NO_CHANGE.
          Write scratchpad. Skip to step 11.
     6. Moderate history (2.0–2.99): DECREASE_CPM unless prev failed → NO_CHANGE.
     7. Weak history: Rule A → B → C → D. First match only.
     8. Position gate: position ≤ 10 and action = PAUSE → override now.
     9. Strong history self-check: action = DECREASE_CPM or PAUSE and strong
        history confirmed → override to NO_CHANGE or INCREASE_CPM before output.
    10. CPM floor check: action = DECREASE_CPM and CPM ≤ cpm_floor → NO_CHANGE.
    11. Previous rec evaluation. Apply learning from user overrides.
    12. Confidence calculation.
    13. Write scratchpad line (if not already written).
    13B. Self-check: IF POS1_BLOCK:YES AND FINAL_ACTION = INCREASE_CPM →
      this is an error. Correct FINAL_ACTION to NO_CHANGE before proceeding.
    14. Write explanation in mandatory 6-part pipe-separated format.
    15. After ALL scratchpad lines complete → write JSON array.
    16. Final check: each JSON action = scratchpad FINAL_ACTION.
        Any mismatch → fix JSON before returning.
         
    """
    
    
    # ============================================================
    # MAIN PIPELINE
    # ============================================================
    
    # Normalize campaign_name_map keys to string — prevents int/str mismatch
    campaign_name_map_str = {str(k): v for k, v in campaign_name_map.items()}
    
    all_suggestions = []
    
    for campaign_id, campaign_group_df in aggregated_df.groupby("campaign_id"):
        # if(campaign_id != '436744' ):
        #     continue
        
        time.sleep(3)
        suggested_keyword_query = f"""
        SELECT 
            REPLACE(ks.keyword, ' ', '_') AS suggested_value,
            MAX(ks.searches) AS total_searches,
            MAX(ks.weighted_score) AS weighted_score,
            BOOL_OR(ks.is_brand_keyword) AS is_brand
        FROM voylla."Blinkit_keyword_suggestions" ks
        GROUP BY ks.keyword
        ORDER BY weighted_score DESC;
        """
    
        keyword_pool_df = pd.read_sql(suggested_keyword_query, engine)
    #     print(keyword_pool_df)
    
        if campaign_id not in sufficient_campaigns:
            continue
    
        campaign_id_str = str(campaign_id)
        campaign_name   = campaign_name_map_str.get(campaign_id_str, "")
    
        print(f"\n🚀 Processing campaign {campaign_id_str} — {campaign_name}")
    
        data_for_llm = campaign_group_df.to_dict(orient="records")
    
        # inject campaign_name into each row so LLM can read it directly
        for row in data_for_llm:
            row["campaign_id"]   = campaign_id_str
            row["campaign_name"] = campaign_name
            targeting_key = str(row.get("targeting", "")).strip().lower().replace(" ", "_")
            prev = build_previous_context(history_df, campaign_id_str, targeting_key)
    
            row["previous_summary"] = prev["previous_summary"]
    
            row["previous_history"] = prev["previous_history"]

            # === Python-side pre-computations (injected as hard facts for LLM) ===

            # 1. Sufficiency booleans (pre-computed so LLM cannot hallucinate them)
            try:
                _sp7   = float(row.get("spend_7d")  or 0)
                _sp15  = float(row.get("spend_15d") or 0)
                _sp30  = float(row.get("spend_30d") or 0)
            except (TypeError, ValueError):
                _sp7 = _sp15 = _sp30 = 0.0

            row["is_7d_sufficient"]  = _sp7  >= 500
            row["is_15d_sufficient"] = _sp15 >= 1000
            row["is_30d_sufficient"] = _sp30 >= 2000

            if   row["is_7d_sufficient"] and row["is_15d_sufficient"] and row["is_30d_sufficient"]:
                row["tier"] = 1
            elif not row["is_7d_sufficient"] and not row["is_15d_sufficient"] and not row["is_30d_sufficient"]:
                row["tier"] = 3
            else:
                row["tier"] = 2

            # 2. ZOMBIE KEYWORD detection (high CPM + top position + zero traffic)
            try:
                _cpm   = float(row.get("current_cpm") or 0)
                _imp   = float(row.get("impressions") or 0)
                _pos   = float(row.get("position") or 999)
                row["zombie_keyword_flag"] = (
                    _cpm  > 300  and
                    _imp  < 50   and
                    _sp7  < 100  and
                    _sp15 < 200  and
                    _pos  <= 5
                )
            except (TypeError, ValueError):
                row["zombie_keyword_flag"] = False

            # 3. POSITIVE ROAS SIGNAL — keyword had returns at some point
            #    Even if the window is insufficient for gate decisions,
            #    any positive ROAS is real evidence the keyword can convert.
            try:
                _r7  = float(row.get("roas_7d")  or 0)
                _r15 = float(row.get("roas_15d") or 0)
                _r30 = float(row.get("roas_30d") or 0)
                row["has_positive_roas_signal"] = (
                    (_r30 >= 2.0 and _sp30 > 0) or
                    (_r15 >= 2.0 and _sp15 > 0)
                )
            except (TypeError, ValueError):
                row["has_positive_roas_signal"] = False

            # 4. SUFFICIENT BURN WITH ZERO ROAS — spent enough to confirm failure
            #    Justified DECREASE even on TIER 3 keywords (not zombie, no sales)
            try:
                row["sufficient_burn_no_roas"] = (
                    _sp30 >= 500  and
                    _r7  == 0.0   and
                    _r15 == 0.0   and
                    _r30 == 0.0   and
                    not row["has_positive_roas_signal"]
                )
            except (TypeError, ValueError):
                row["sufficient_burn_no_roas"] = False

            # 5. DYNAMIC CPM FLOOR + SEARCH VOLUME ANALYSIS
            try:
                _exact_min  = float(row.get("exact_min")  or 0)
                _min_bid    = float(row.get("min_bid")    or 0)
                _ks_raw  = row.get("keyword_searches")
                try:
                    _searches = 0.0 if (_ks_raw is None or (isinstance(_ks_raw, float) and math.isnan(_ks_raw))) else float(_ks_raw)
                except (TypeError, ValueError):
                    _searches = 0.0
                _cpm        = float(row.get("current_cpm") or 0)

                # Dynamic CPM floor — replaces hardcoded 223
                row["cpm_floor"] = round(_exact_min, 2) if _exact_min > 0 else None

                # How many times over the floor is the current CPM?
                row["cpm_to_min_ratio"] = (
                    round(_cpm / _exact_min, 2) if _exact_min > 0 else None
                )

                # Search volume tier
                if   _searches <= 0:    row["search_volume_tier"] = "UNKNOWN"
                elif _searches < 100:   row["search_volume_tier"] = "DEAD"
                elif _searches < 400:   row["search_volume_tier"] = "LOW"
                elif _searches < 2000:  row["search_volume_tier"] = "MEDIUM"
                else:                   row["search_volume_tier"] = "HIGH"

                # PAUSE candidate: dead category + zero ROAS + near-zero spend
                row["search_pause_flag"] = bool(
                    _searches > 0 and
                    _searches < 100 and
                    float(row.get("roas_7d")  or 0) == 0.0 and
                    float(row.get("roas_15d") or 0) == 0.0 and
                    float(row.get("roas_30d") or 0) == 0.0 and
                    float(row.get("spend_7d") or 0) < 50
                )

                # Decrease candidate: LOW/DEAD search volume + zero ROAS across ALL windows
                _r7_ls  = float(row.get("roas_7d")  or 0)
                _r15_ls = float(row.get("roas_15d") or 0)
                _r30_ls = float(row.get("roas_30d") or 0)
                row["low_search_zero_roas"] = bool(
                    _searches > 0 and
                    _searches < 400 and
                    _r7_ls  == 0.0 and
                    _r15_ls == 0.0 and
                    _r30_ls == 0.0 and
                    not row.get("has_positive_roas_signal", False)
                )

            except (TypeError, ValueError):
                row["cpm_floor"]           = None
                row["cpm_to_min_ratio"]    = None
                row["search_volume_tier"]  = "UNKNOWN"
                row["search_pause_flag"]   = False
                row["low_search_zero_roas"]= False


#         campaign_history_df = history_df[
#             history_df["campaign_id"].astype(str) == campaign_id_str
#         ].copy()
        
#         print(campaign_history_df)
    
        user_message = f"""
             ⚠️ HARD RULE (non-negotiable): If current_cpm ≤ cpm_floor (or ≤ ₹200 if cpm_floor is null) and action=DECREASE_CPM, then override -> action = NO_CHANGE.
        ⚠️ HIGH CPM BLEED RULE: If current_cpm > ₹350 AND any sufficient ROAS window < 2.0 AND position ≠ 1 → DECREASE_CPM (overrides strong history gate). Mark HIGH_CPM_BLEED:YES in scratchpad.
   
        ⚠️ action field = LLM Insight action (not Rule Decision).
        ⚠️ TWO hard constraints enforced by Python AFTER your response:
           1. current_cpm <= cpm_floor -> DECREASE_CPM blocked by code.
           2. position = 1 AND INCREASE_CPM -> blocked by code.
           You do NOT need to self-censor around these - Python handles them.

           LLM INSIGHT = COMPLETELY FREE. No tier rules. No position guards.
           Think like a senior performance marketer. Make the call YOU believe in.
           Position 1 with 100 searches/month is NOT worth protecting - say so.
           Thin data with zero ROAS may still signal a bad keyword - say so.
           Strong 30d ROAS with a recent dip IS worth protecting - say so.
           One sentence. Be direct.

-> Each row contains two fields:
           "previous_summary" → copy this verbatim into the explanation "Previous:" section.
           "previous_history" → use this list for Step 5 decision-making (newest first).
        -> Do NOT repeat an action flagged with ⚠️ LOOP in previous_summary.
        -> Do NOT cross-reference history between keywords.

        CURRENT DATA (7-day aggregated with 15-day and 30-day ROAS and spend):
        {json.dumps(data_for_llm)}

        KEYWORD POOL (for alternative keywords when action = PAUSE):
        {json.dumps(keyword_pool_df.to_dict(orient='records'))}
        
        ═══════════════════════════════════════════════
        DECISION PROCESS (MANDATORY — follow in this order)
        ═══════════════════════════════════════════════

        For each keyword:

        1. Compute Rule Decision using ALL system rules. 
           This is for transparency only — it does NOT set the action field.

        2. Compute LLM Insight:
           - IGNORE ALL rules completely
           - No CPM floor, no ROAS thresholds, no tier logic
           - Think like a human expert optimizing long-term growth
           - Focus on trends, momentum, and scaling opportunity
           - This SETS the action field.

        3. Apply TWO hard overrides to LLM Insight (only these, nothing else):
           - If current_cpm ≤ cpm_floor (or ≤ ₹200 if null) AND LLM Insight = DECREASE_CPM → action = NO_CHANGE
             (INCREASE_CPM and NO_CHANGE remain valid at any CPM level)
           - If most_viewed_position = 1 AND LLM Insight = INCREASE_CPM → action = NO_CHANGE

        4. FINAL_ACTION = LLM Insight after overrides. NOT Rule Decision.

        5. If Rule Decision and LLM Insight differ, note it in explanation.

        ═══════════════════════════════════════════════
        EXPLANATION FORMAT (MANDATORY — INCLUDE BOTH)
        ═══════════════════════════════════════════════

        Write the explanation as a single string in this exact format:

        7-Day: Spend ₹[exact]. ROAS [exact]. Position [exact]. CPM ₹[current_cpm]. Sufficiency: [Met ₹500 / Not met ₹500]. | 
        15-Day: Spend ₹[exact]. ROAS [exact]. Sufficiency: [Met ₹1000 / Not met ₹1000]. Trend vs 7-day: [improving / declining / stable / insufficient]. | 
        30-Day: Spend ₹[exact]. ROAS [exact]. Sufficiency: [Met ₹2000 / Not met ₹2000]. Strength: [Strong ≥3.0 / Moderate 2.0–2.99 / Weak <2.0 / Insufficient data]. | 
        
        Rule Decision: [Action suggestion] — [1–2 sentence reasoning] |
        
        LLM Insight: [Action suggestion] — [1–2 sentence reasoning ignoring all rules, focusing on trends, momentum, or opportunity.] | 

        Previous: [copy previous_summary from this keyword's row verbatim] | 

        Final Recommendation: [LLM Insight action] — [1-line justification]

        ═══════════════════════════════════════════════

        Write scratchpad lines first. Then return ONLY the JSON array.
        
        ⚠️ FINAL CHECK (MANDATORY):
        Before returning JSON, ensure EVERY object contains "confidence".
        If missing → FIX the object before returning.
        
        No preamble, no commentary, no markdown fences.
        ═══════════════════════════════════════════════
        CONFIDENCE ENFORCEMENT (CRITICAL — NO EXCEPTIONS)
        ═══════════════════════════════════════════════

        For EVERY keyword:

        1. You MUST calculate a confidence score.

        2. Confidence MUST:
           - Be a decimal between 0.70 and 0.95
           - Never be 0, null, or missing
           - Never be 1.0

        3. You MUST include confidence in:
           - Scratchpad line
           - JSON output

        4. Scratchpad format MUST end with:
           | CONFIDENCE:[value]

        5. JSON "confidence" MUST EXACTLY MATCH scratchpad value.

        6. If confidence is missing → output is INVALID.

        7. If unsure → default to 0.75 (never skip).

        ═══════════════════════════════════════════════

        Instructions:
        If current_cpm is <= cpm_floor (or <= 200 if null) and action is decrease_cpm, then action should be no change.

        1. Write scratchpad lines first (one per keyword).
        2. Then output the JSON array with ALL 10 fields per object:
           campaign_id, targeting, action, explanation, campaign_name,
           cpm_change, confidence, alternative_keywords, current_cpm, campaign_budget.
        3. confidence must match the CONFIDENCE value from your scratchpad line.
        4. No missing fields. No null values. No markdown fences.
        5.Return JSON array. Each object MUST include "confidence".
        
        Then output the JSON array.

        Each JSON object MUST contain ALL of these fields EXACTLY:

        {{
          "campaign_id": "",
          "targeting": "",
          "action": "INCREASE_CPM | DECREASE_CPM | NO_CHANGE | PAUSE",
          "explanation": "",
          "campaign_name": "",
          "cpm_change": 0,
          "confidence": 0.80,
          "alternative_keywords": [],
          "current_cpm": 0,
          "campaign_budget": 0
        }}

        CRITICAL:
        - "confidence" field is MANDATORY
        - It must be present in EVERY object
        - It must be a decimal between 0.70 and 0.95
        - It MUST match the CONFIDENCE value in scratchpad
        - Missing confidence = INVALID OUTPUT
        """

        raw_response = ""
    
        try:
#             response = llm.invoke([
#                 SystemMessage(content=SYSTEM_PROMPT),
#                 HumanMessage(content=user_message)
#             ] , config={"max_tokens": 8000})
    
#             raw_response = response.content

            for _attempt in range(3):
                try:
                    response = llm.messages.create(
                        model=MODEL_NAME,
                        max_tokens=6000,
                        temperature=LLM_TEMPERATURE,
                        system=[{"type": "text", "text": SYSTEM_PROMPT, "cache_control": {"type": "ephemeral"}}],
                        messages=[
                            {"role": "user", "content": user_message}
                        ]
                    )
                    break
                except Exception as _e:
                    if "rate_limit" in str(_e).lower() and _attempt < 2:
                        import time as _t; _t.sleep(60 * (_attempt + 1))
                    else:
                        raise
    
            raw_response=response.content[0].text

            print(f"📝 Raw response preview:\n{raw_response[:600]}\n...")
    
            suggestion_response = extract_json(raw_response)
            print("🔍 Sample JSON row:", json.dumps(suggestion_response[0], indent=2))  # add this
    
            # patch campaign_id and campaign_name on every row — guaranteed correct
            # preserve all other columns exactly as returned by LLM
            # Also: re-inject Python-computed flags so save_llm_action can enforce overrides
            row_lookup = {
                str(r.get("targeting", "")).strip().lower().replace(" ", "_"): r
                for r in data_for_llm
            }
            for row in suggestion_response:
                row["campaign_id"]   = campaign_id_str
                row["campaign_name"] = campaign_name
                tkey = str(row.get("targeting", "")).strip().lower().replace(" ", "_")
                orig = row_lookup.get(tkey, {})
                # Re-inject Python ground-truth flags (LLM cannot override these in Python)
                row["zombie_keyword_flag"]      = orig.get("zombie_keyword_flag",      False)
                row["has_positive_roas_signal"] = orig.get("has_positive_roas_signal", False)
                row["sufficient_burn_no_roas"]  = orig.get("sufficient_burn_no_roas",  False)
                row["tier"]                     = orig.get("tier",                     3)
                row["keyword_searches"]         = orig.get("keyword_searches",         None)
                row["kw_weighted_score"]        = orig.get("kw_weighted_score",        None)
                row["search_volume_tier"]       = orig.get("search_volume_tier",       "UNKNOWN")
                row["low_search_zero_roas"]     = orig.get("low_search_zero_roas",     False)
                row["search_pause_flag"]        = orig.get("search_pause_flag",        False)
                row["cpm_floor"]               = orig.get("cpm_floor",               None)
                row["most_viewed_position"]    = orig.get("most_viewed_position",    99)
                # Lock the Searches line in explanation to pre-computed value
                _correct_tier  = str(orig.get("search_volume_tier") or "UNKNOWN")
                _correct_count = _safe_int(orig.get("keyword_searches"), 0)
                _searches_str  = f"Searches: {_correct_tier} ({_correct_count})"
                _expl_fix = row.get("explanation", "")

                # Step 1: Replace any existing Searches/Search volume pattern
                _expl_fix = re.sub(
                    r"(?:Searches|Search volume):\s*[^.|]*?\([^)]*\)",
                    _searches_str,
                    _expl_fix
                )

                # Step 2: If LLM omitted Searches entirely, INSERT it after first "CPM ..."
                # Target pattern: "CPM ₹X" or "CPM ₹X (floor ₹Y)." — insert "Searches: ..." right after
                if "Searches:" not in _expl_fix:
                    # Insert before "Sufficiency:" in the 7-day section
                    _expl_fix_new = re.sub(
                        r"(CPM\s*[^.|]*?\.)\s*(Sufficiency)",
                        rf"\1 {_searches_str}. \2",
                        _expl_fix,
                        count=1
                    )
                    if _expl_fix_new != _expl_fix:
                        _expl_fix = _expl_fix_new
                    else:
                        # Fallback: prepend to explanation if structure unexpected
                        _expl_fix = f"{_searches_str}. | " + _expl_fix
                row["explanation"] = _expl_fix

            print(f"✅ {len(suggestion_response)} keywords processed")
    
            all_suggestions.extend(suggestion_response)
    
            save_llm_action(engine, suggestion_response,brand)
    
        except ValueError as e:
            print(f"❌ JSON parse error for campaign {campaign_id_str}: {e}")
            if raw_response:
                print(f"   Raw response snippet: {raw_response[:400]}")
            continue
    
        except Exception as e:
            print(f"❌ Unexpected error for campaign {campaign_id_str}: {e}")
            if raw_response:
                print(f"   Raw response snippet: {raw_response[:400]}")
            continue
    
    print(f"\n📊 Total keywords processed: {len(all_suggestions)}")
    
    
    # ============================================================
    # QUICK SANITY CHECK — print summary table
    # ============================================================
    #     all_suggestions=[]
#     ─────────────────────────────────────────────
#     STEP 1: Handle INSUFFICIENT campaigns
#     (just for alternative keyword suggestions)
#     ─────────────────────────────────────────────
    for campaign_id in insufficient_campaigns:
        
        campaign_group_df = aggregated_df[aggregated_df["campaign_id"] == campaign_id]
        campaign_name = campaign_name_map_str.get(str(campaign_id), "")
        data_for_llm = campaign_group_df.to_dict(orient="records")
    #     print(data_for_llm)
        total_spend = campaign_spend_df[
            campaign_spend_df["Campaign ID"] == campaign_id
        ]["campaign_spend"].values[0]
        
        
        
        suggested_keyword_query = f"""
        SELECT 
            REPLACE(ks.keyword, ' ', '_') AS suggested_value,
            MAX(ks.searches) AS total_searches,
            MAX(ks.weighted_score) AS weighted_score,
            BOOL_OR(ks.is_brand_keyword) AS is_brand
        FROM voylla."Blinkit_keyword_suggestions" ks
        GROUP BY ks.keyword
        ORDER BY weighted_score DESC;
        """
    
        keyword_pool_df = pd.read_sql(suggested_keyword_query, engine)
    #     print(keyword_pool_df)
    
        print(f"\n⚠️  INSUFFICIENT campaign {campaign_id} | 7d campaign spend ₹{total_spend:.0f} — generating keyword suggestions only")
    
        insufficient_prompt = f"""
        You are a performance marketing expert analyzing Blinkit ad campaigns.
    
        CONTEXT:
        - This campaign has a total 7-day spend of ₹{total_spend:.0f}, which is below the ₹500 threshold.
        - All keywords in this campaign are classified as INSUFFICIENT DATA.
        - Do NOT recommend PAUSE or INCREASE_CPM. Action must always be INSUFFICIENT_DATA.
        - Your only task: suggest 2–3 semantically similar alternative keywords for EACH keyword below.
    
        RULES FOR ALTERNATIVE KEYWORDS:
        1. Suggest keywords semantically similar to the targeting keyword
        2. DO NOT suggest the same keyword being analyzed
        3. Each keyword MUST receive DIFFERENT alternative suggestions
        4. No duplicates across the entire batch
        5. If similarity is low, still suggest the closest 2 keywords from the keyword pool. Never return an empty array.
    
        OUTPUT FORMAT:
        Return ONLY a valid JSON array. Each object must have this exact structure:
        
          "campaign_id": "{campaign_id}",
          "campaign_name": "{campaign_name}",
          "targeting": "keyword name",
          "action": "INSUFFICIENT_DATA",
          "cpm_change": null,
          "confidence": 0.5,
          "explanation": "7-day campaign spend ₹{total_spend:.0f}. Below ₹500 threshold. INSUFFICIENT data. Action deferred — suggesting alternative keywords to explore.",
          "alternative_keywords": ["kw1", "kw2", "kw3","k4"],
          "current_cpm":current_cpm,
          "campaign_budget":campaign_budget
          
        KEYWORD POOL (for alternatives keywords):
        {keyword_pool_df.to_dict(orient='records')}
        
    
        CURRENT KEYWORDS IN THIS CAMPAIGN:
        {json.dumps(data_for_llm)}
    
        Return ONLY the JSON array.

        Instructions:
        1. Write scratchpad lines first (one per keyword).
        2. Then output the JSON array with ALL fields per object:
           campaign_id, targeting, action, explanation, campaign_name,
           cpm_change, confidence, alternative_keywords,current_cpm,campaign_budget.
        3. confidence must match the CONFIDENCE value from your scratchpad line.
        4. No missing fields. No null values. No markdown fences.
        """
    
#         response = llm.invoke(insufficient_prompt).content
    #     print(response)
    
        for _attempt in range(3):
            try:
                response = llm.messages.create(
                    model=MODEL_NAME,
                    max_tokens=8000,
                    temperature=LLM_TEMPERATURE,
                    messages=[
                        {"role": "user", "content": insufficient_prompt}
                    ]
                ).content[0].text
                break
            except Exception as _e:
                if "rate_limit" in str(_e).lower() and _attempt < 2:
                    import time as _t; _t.sleep(60 * (_attempt + 1))
                else:
                    raise
        
        suggestion_response = extract_json(response)
    
        for row in suggestion_response:
            if not isinstance(row, dict):
                continue 
            row["campaign_id"] = campaign_id
            row["campaign_name"] = campaign_name
            row["action"] = "INSUFFICIENT DATA"  # force correct action
            row["explanation"]="INSUFFICIENT DATA"
    
    #     print(suggestion_response)
        save_llm_action(engine, suggestion_response,brand)
        all_suggestions.extend(suggestion_response)

    
    
    if all_suggestions:
        import pandas as pd
        results_df = pd.DataFrame(all_suggestions)
    
        print("\n📋 Action distribution:")
        print(results_df["action"].value_counts().to_string())
    
        print("\n📋 Campaign name check (should have no blanks):")
        blank_names = results_df[results_df["campaign_name"] == ""]
        
        if blank_names.empty:
            print("   ✅ All campaign names populated")
        else:
            print(f"   ⚠️  {len(blank_names)} rows with blank campaign name:")
            print(blank_names[["campaign_id", "targeting"]].to_string())
            
    
        # print("\n📋 Sample output:")
        # print(results_df[["campaign_id", "campaign_name", "targeting",
        #                    "action", "cpm_change", "confidence"]].to_string())    

In [ ]:
all_suggestions

In [ ]:
xyz

In [ ]:
import datetime
st=datetime.datetime.fromtimestamp(time.time()).strftime('%Y-%m-%d %H:%M:%S')

#Script Details
SD={
    'Script_Name':'Blinkit_actions_llm_marketing.ipynb',
    'Output':'Table',
    'Table Name':'"Blinkit_actions_llm"',
    'Sheet_id':'-',
    'Report_Name':'-',
    'Updated_at':st
}

SD = pd.DataFrame([SD])

SD.to_sql('python_log', engine, schema='voylla', if_exists='append', index=False)